# Pre-M0.4 — IQ Broadcasting and Vectorized Operations

## Unit Objective

Understand how NumPy broadcasting enables concise, efficient element-wise operations on multi-dimensional IQ arrays, and learn to recognize both compatible and incompatible shape combinations.

## What You Will Learn

- How element-wise operations and scalar multiplication work on NumPy arrays
- The rules of broadcasting: trailing dimensions, size-1 axes, and compatible shapes
- Why incompatible shapes raise errors and how to diagnose them
- How to perform channel-wise scaling on IQ data with a `(1, 2, 1)` scale tensor
- Why vectorized operations outperform Python loops and how to verify equivalence
- How an injected axis swap breaks broadcasting semantics

## IQ Data Convention (Review)

Throughout this notebook, IQ data follows the canonical layout:

| Axis | Meaning | Size |
|------|---------|------|
| 0 | Examples | N |
| 1 | I/Q (0=I, 1=Q) | 2 |
| 2 | Time samples | L |

- **dtype**: `np.float32`
- **SEED**: 42 for reproducibility
- Canonical shape: `X.shape == (N, 2, L)`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(42)

N = 8
L = 200

X = np.random.randn(N, 2, L).astype(np.float32)

print(f"X.shape = {X.shape}")
print(f"X.dtype = {X.dtype}")

---

## 1. Element-Wise Operations and Scalar Multiplication

NumPy applies arithmetic operators element-by-element. A scalar operand is broadcast to match every element of the array.

In [ ]:
doubled = X * 2.0
shifted = X + 1.0
negated = -X

print(f"(X * 2).shape = {doubled.shape}")
print(f"Original dtype preserved: {doubled.dtype == X.dtype}")
print(f"doubled[0, 0, :3] = {doubled[0, 0, :3]}")
print(f"X[0, 0, :3] * 2   = {X[0, 0, :3] * 2.0}")

### Small Example

A 1-D array of shape `(L,)` added to a 3-D array of shape `(N, 2, L)` broadcasts along axes 0 and 1:

In [ ]:
time_axis = np.arange(L, dtype=np.float32)
result = X + time_axis  # (N, 2, L) + (L,)
print(f"(N,2,L) + (L,) => shape {result.shape}")

---

## 2. Broadcasting Rules

Broadcasting compares shapes **from the trailing dimension** (rightmost) to leftward. Two dimensions are compatible when:

1. They are equal, **or**
2. One of them is `1`

A dimension of size `1` is **replicated** to match the other.

### Compatible Shapes

For IQ channel-wise scaling, we want to scale I and Q channels independently across all examples and time samples.

```
X           : (N, 2, L)
channel_scales : (1, 2, 1)
```

Trailing dimension: `L` vs `1` → compatible (broadcast)

Middle dimension: `2` vs `2` → compatible (equal)

Leading dimension: `N` vs `1` → compatible (broadcast)

In [ ]:
channel_scales = np.array([[[1.5], [0.5]]], dtype=np.float32)
print(f"channel_scales.shape = {channel_scales.shape}")
print(f"channel_scales = \n{channel_scales}")

scaled = X * channel_scales
print(f"\nX * channel_scales => shape {scaled.shape}")

# Verify: I channel scaled by 1.5, Q channel scaled by 0.5
assert np.allclose(scaled[:, 0, :], X[:, 0, :] * 1.5)
assert np.allclose(scaled[:, 1, :], X[:, 1, :] * 0.5)
print("Channel-wise scaling verified.")

### Why Broadcasting Works Here

| Axis | X | channel_scales | Rule | Result |
|------|---|----------------|------|--------|
| 2 (trailing) | L | 1 | size-1 broadcast | L |
| 1 | 2 | 2 | equal | 2 |
| 0 (leading) | N | 1 | size-1 broadcast | N |

Result shape: `(N, 2, L)` — every example and every time sample gets the same per-channel scale.

### Incompatible Shapes

If the I/Q axis sizes don't match and neither is 1, broadcasting fails.

In [ ]:
bad_scales = np.array([1.0, 0.5, 0.3], dtype=np.float32)  # shape (3,)
print(f"bad_scales.shape = {bad_scales.shape}")
print(f"X.shape = {X.shape}")
print("\nAttempting X * bad_scales...")
try:
    result_bad = X * bad_scales
except ValueError as e:
    print(f"ValueError: {e}")
    print("\nThe trailing dimension (3) does not match L and is not 1.")
    print("Broadcasting cannot replicate 3 into L.")

---

## 3. Vectorization vs Python Loops

A Python `for` loop over time samples is slow. The equivalent NumPy vectorized expression runs in optimized C.

In [ ]:
scale_i = 1.5
scale_q = 0.5

# Python loop approach
X_loop = np.empty_like(X)
for i in range(N):
    for t in range(L):
        X_loop[i, 0, t] = X[i, 0, t] * scale_i
        X_loop[i, 1, t] = X[i, 1, t] * scale_q

# Vectorized approach
scales_vec = np.array([[[scale_i]], [[scale_q]]], dtype=np.float32)
X_vec = X * scales_vec

print(f"Loop result shape:   {X_loop.shape}")
print(f"Vector result shape: {X_vec.shape}")
print(f"Results match: {np.allclose(X_loop, X_vec)}")

---

## 4. Relating Injected Axis Swap to Broadcasting

If someone accidentally swaps the I/Q and time axes, the broadcasting semantics change. The `(1, 2, 1)` scale tensor would now target the wrong axes.

In [ ]:
X_swapped_preview = np.transpose(X, (0, 2, 1))  # shape (N, L, 2)
print(f"X_swapped_preview.shape = {X_swapped_preview.shape}")
print(f"Original X.shape = {X.shape}")

# Attempting the same broadcast with swapped axes
try:
    bad_broadcast = X_swapped_preview * channel_scales
    print(f"Shape after broadcast: {bad_broadcast.shape}")
    print("Warning: no error, but semantics are WRONG.")
    print("(1,2,1) broadcast on (N,L,2) means L is broadcast, 2 matches 2, N is broadcast.")
    print("The scale now applies along axis-2 (I/Q dim of swapped array) which is WRONG.")
except ValueError as e:
    print(f"ValueError: {e}")

print("\nKey insight: An axis swap does not always cause an error.")
print("It can silently produce incorrect results. This is the semantic danger.")

---

## Student Exercises

Complete each exercise before proceeding.

### Exercise 1

Create a scale array `scales` of shape `(1, 2, 1)` that multiplies I by 2.0 and Q by -1.0. Apply it to `X` and verify that the I channel is doubled and the Q channel is negated.

In [ ]:
# STUDENT ATTEMPT
# TODO: Create scales and apply to X

# YOUR CODE HERE
pass

### Exercise 2

Write a Python loop that applies the same per-channel scaling (I × 2.0, Q × −1.0) to `X`. Compare results with `np.allclose`.

In [ ]:
# STUDENT ATTEMPT
# TODO: Implement loop version and compare

# YOUR CODE HERE
pass

---

## OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

### Exercise 1 Solution

In [ ]:
scales = np.array([[[2.0], [-1.0]]], dtype=np.float32)
scaled_ex1 = X * scales
assert np.allclose(scaled_ex1[:, 0, :], X[:, 0, :] * 2.0)
assert np.allclose(scaled_ex1[:, 1, :], X[:, 1, :] * -1.0)
print("Exercise 1 verified.")

### Exercise 2 Solution

In [ ]:
X_loop_ex2 = np.empty_like(X)
for i in range(N):
    for t in range(L):
        X_loop_ex2[i, 0, t] = X[i, 0, t] * 2.0
        X_loop_ex2[i, 1, t] = X[i, 1, t] * -1.0

assert np.allclose(X_loop_ex2, scaled_ex1)
print("Exercise 2 verified: loop matches vectorized.")

---

## Pass Criterion Challenge

Complete all three gates below to pass this notebook.

---

## PC-1 — Independent Axis Explanation

Answer each question in a Markdown cell. Be precise and concise.

### STUDENT ATTEMPT

**Q1**: In `X.shape == (N, 2, L)`, what does axis 0 represent?

**Q2**: What does axis 1 represent, and which index is I, which is Q?

**Q3**: What does axis 2 represent?

**Q4**: In `channel_scales.shape == (1, 2, 1)`, what does the middle axis (size 2) correspond to in `X`?

**Q5**: Why does the leading `1` in `channel_scales` not cause a shape mismatch with `N` in `X`?

**Q6**: If you have a scale array of shape `(2,)`, why does `X * scale_array` raise a `ValueError`?

In [ ]:
AXES_EXPLANATION_VERIFIED = False
# Student or instructor must manually change this to True
# after verifying all 6 answers are correct.

### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

**Q1**: Axis 0 represents individual examples (waveforms/observations). There are N examples.

**Q2**: Axis 1 represents the I/Q components. Index 0 is I (in-phase), index 1 is Q (quadrature).

**Q3**: Axis 2 represents time samples. Each waveform has L discrete samples.

**Q4**: The middle axis (size 2) in `channel_scales` corresponds to the I/Q axis (axis 1) in `X`. The two values scale I and Q independently.

**Q5**: The leading `1` is compatible with `N` because broadcasting rule says: if one dimension is 1, it is replicated. So `1` expands to `N` along axis 0.

**Q6**: `(N, 2, L) * (2,)` → trailing dimension: `L` vs `2`. Neither is 1, and they are not equal → incompatible.

---

## PC-2 — Injected Axis Swap

A colleague swapped axes 1 and 2 in your IQ array. You must detect, diagnose, and correct the error.

In [ ]:
# Create original IQ data
np.random.seed(42)
N_pc2 = 8
L_pc2 = 200

X_original = np.random.randn(N_pc2, 2, L_pc2).astype(np.float32)
print(f"X_original.shape = {X_original.shape}")

In [ ]:
# Inject the axis swap error
X_swapped = np.transpose(X_original, (0, 2, 1))
print(f"X_swapped.shape = {X_swapped.shape}")
print("\nERROR INJECTED: axis 1 and axis 2 have been swapped.")

### STUDENT ATTEMPT

**Step 1**: Inspect `X_swapped.shape`. What shape do you see?

**Step 2**: Which axis now holds the I/Q components? Which axis holds time samples?

**Step 3**: Explain why this is wrong for the IQ contract.

**Step 4**: Write the correction and produce `X_fixed`.

In [ ]:
# STUDENT ATTEMPT
# TODO: Write your correction here

# YOUR CODE HERE
X_fixed = None  # Replace with your correction

### Validation

In [ ]:
axis_swap_corrected = (
    X_fixed.shape == X_original.shape
    and X_fixed.dtype == X_original.dtype
    and np.array_equal(X_fixed, X_original)
)
assert X_fixed.shape == X_original.shape, f"Expected shape {X_original.shape}, got {X_fixed.shape}"
assert X_fixed.dtype == X_original.dtype, f"Expected dtype {X_original.dtype}, got {X_fixed.dtype}"
assert np.array_equal(X_fixed, X_original), "X_fixed does not match X_original"
print("PC-2 PASSED: Axis swap corrected. X_fixed matches X_original.")

### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

In [ ]:
X_fixed_solution = np.transpose(X_swapped, (0, 2, 1))
assert X_fixed_solution.shape == X_original.shape
assert X_fixed_solution.dtype == X_original.dtype
assert np.array_equal(X_fixed_solution, X_original)
print("Solution verified: np.transpose(X_swapped, (0, 2, 1)) recovers X_original.")

---

## PC-3 — IQ Power Agreement

Compute the mean power of IQ signals using two independent methods and verify they agree.

In [ ]:
np.random.seed(42)
N_pc3 = 8
L_pc3 = 200

X_power = np.random.randn(N_pc3, 2, L_pc3).astype(np.float32)

I = X_power[:, 0, :]  # shape (N, L)
Q = X_power[:, 1, :]  # shape (N, L)

print(f"I.shape = {I.shape}, Q.shape = {Q.shape}")

### STUDENT ATTEMPT

**Method A**: Compute `P_iq = np.mean(I**2 + Q**2)`

**Method B**: Compute `z = I + 1j*Q; P_complex = np.mean(np.abs(z)**2)`

Both should produce a scalar representing the average power.

In [ ]:
# STUDENT ATTEMPT
# TODO: Implement Method A and Method B

# YOUR CODE HERE
P_iq = None
P_complex = None

### Validation

In [ ]:
POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

power_consistency = np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
assert power_consistency, f"Power mismatch: P_iq={P_iq}, P_complex={P_complex}"

print(f"P_iq     = {P_iq:.8f}")
print(f"P_complex = {P_complex:.8f}")
print(f"PC-3 PASSED: Power methods agree within tolerances.")

### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

In [ ]:
P_iq_solution = np.mean(I**2 + Q**2)
z = I + 1j * Q
P_complex_solution = np.mean(np.abs(z)**2)

assert np.allclose(P_iq_solution, P_complex_solution, rtol=POWER_RTOL, atol=POWER_ATOL)
print(f"Solution verified: P_iq={P_iq_solution:.8f}, P_complex={P_complex_solution:.8f}")

---

## Automatic Validations

In [ ]:
np.random.seed(42)

# Reconstruct for validation
X_val = np.random.randn(N, 2, L).astype(np.float32)

# Gate: vectorized vs loop equivalence
scales_val = np.array([[[2.0], [-1.0]]], dtype=np.float32)
X_vec_val = X_val * scales_val

X_loop_val = np.empty_like(X_val)
for i in range(N):
    for t in range(L):
        X_loop_val[i, 0, t] = X_val[i, 0, t] * 2.0
        X_loop_val[i, 1, t] = X_val[i, 1, t] * -1.0

vec_loop_match = np.allclose(X_vec_val, X_loop_val)
print(f"Vectorized == Loop: {vec_loop_match}")

# Gate: axis swap round-trip
X_rt = np.transpose(np.transpose(X_val, (0, 2, 1)), (0, 2, 1))
print(f"Axis swap round-trip recovers: {np.array_equal(X_rt, X_val)}")

# Gate: power agreement
I_val = X_val[:, 0, :]
Q_val = X_val[:, 1, :]
p1 = np.mean(I_val**2 + Q_val**2)
p2 = np.mean(np.abs(I_val + 1j * Q_val)**2)
print(f"Power agreement: {np.allclose(p1, p2, rtol=1e-5, atol=1e-7)}")

---

## Manual Evaluation: Axis Explanation

An instructor or the student must review the 6 answers in **PC-1** and set `AXES_EXPLANATION_VERIFIED = True` if all are correct.

In [ ]:
print(f"AXES_EXPLANATION_VERIFIED = {AXES_EXPLANATION_VERIFIED}")

pc1_status = "PASS" if AXES_EXPLANATION_VERIFIED else "WAIT"
print(f"PC-1 Status: {pc1_status}")

---

## PASS CRITERION GATE

In [ ]:
pc1_pass = AXES_EXPLANATION_VERIFIED
pc2_pass = (
    X_fixed is not None
    and X_fixed.shape == X_original.shape
    and X_fixed.dtype == X_original.dtype
    and np.array_equal(X_fixed, X_original)
)
pc3_pass = (
    P_iq is not None
    and P_complex is not None
    and np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
)

print(f"PC-1 (Axes Explanation): {"PASS" if pc1_pass else "WAIT"}")
print(f"PC-2 (Axis Swap Fix):     {"PASS" if pc2_pass else "WAIT"}")
print(f"PC-3 (Power Agreement):   {"PASS" if pc3_pass else "WAIT"}")
print()

if pc1_pass and pc2_pass and pc3_pass:
    print("PRE-M0.4 FINAL STATUS: PASS")
else:
    print("PRE-M0.4 FINAL STATUS: WAIT")